In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
import warnings
import xgboost as xgb
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme()

In [2]:
train = pd.read_csv("data/track_k_ecommerce_returns_train.csv")
test = pd.read_csv("data/track_k_ecommerce_returns_test.csv")

# Threshold

Our target seems to have a bimodal distribution. <br>
Let's try to find a threshold to separate the features and using different models on them.

In [3]:
from sklearn.tree import DecisionTreeRegressor

X_threshold = train['price_eur']
y = train['return_rate_pct']

threshold_finder = DecisionTreeRegressor(max_depth=1, random_state=42)
threshold_finder.fit(X_threshold.values.reshape(-1, 1), y)

optimal_threshold = threshold_finder.tree_.threshold[0]
print(f"The optimal split point is: {optimal_threshold:.2f}")

The optimal split point is: 173.96


We're using DecisionTreeRegressor at 1 depth to make the sole most efficient cut of our data

In [4]:
df_low_price = train[train['price_eur'] <= optimal_threshold]
df_high_price = train[train['price_eur'] > optimal_threshold]

X_low = df_low_price.drop(columns=['return_rate_pct'])
y_low = df_low_price['return_rate_pct']

X_high = df_high_price.drop(columns=['return_rate_pct'])
y_high = df_high_price['return_rate_pct']

## Test on RandomForest

Separating the datas into two frames so we can train two models.

In [5]:
#Test on RdForest

model_low_tier = RandomForestRegressor(n_estimators=100, random_state=42)
model_high_tier = RandomForestRegressor(n_estimators=100, random_state=42)

model_low_tier.fit(X_low, y_low)
print("Low_Done")
model_high_tier.fit(X_high, y_high)
print("High_Done")

Low_Done
High_Done


In [8]:
def predict_return_rate(new_data_df, threshold, model_low, model_high):
    #To hold onto the predictions
    predictions = np.zeros(len(new_data_df))
    
    #Sort of boolean list that serve as a filter
    is_low = new_data_df['price_eur'] <= threshold #[True False True ...] True for low tier, False for high tier
    is_high = new_data_df['price_eur'] > threshold #[True False True ...] True for high tier, False for low tier
    
    #Low tier predictions
    if is_low.any():
        #New list with index of the low tiers + their features
        X_new_low = new_data_df.loc[is_low, model_low.feature_names_in_]

        #Predictions[is_low] targets the index of the low tiers from above
            # [True False True ...] => Index 1 and 3
        predictions[is_low] = model_low.predict(X_new_low)
        
    # High tier predictions
    if is_high.any():
        #Same principle as low tier, but for high tier

        X_new_high = new_data_df.loc[is_high, model_high.feature_names_in_]
        predictions[is_high] = model_high.predict(X_new_high)
    
        
    return predictions

final_predictions = predict_return_rate(test, optimal_threshold, model_low_tier, model_high_tier)

#print mse
final_mse = mean_squared_error(test['return_rate_pct'], final_predictions)
print(f"Final MSE on test set: {final_mse:.4f}")


Final MSE on test set: 25.4354


We have a better MSE than previously using RdForest(28.57) <br>
Now let's try to apply that to the previous Nested CV

## Threshold on Nested CV

In [23]:
# Here we are going to compare Ridge, Lasso and ElasticNet, each with its own set of hyperparameters. 
# We will use nested cross-validation to do that, which allows us to get an unbiased estimate of the test performance of the best model.

from xgboost import XGBRegressor as XGBoost


# Gradient Boost
from sklearn.ensemble import GradientBoostingRegressor

#Adaboost
from sklearn.ensemble import AdaBoostRegressor

#SVM
from sklearn import svm


outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)



param_ridge = {"alpha": np.logspace(-2, 4, 20)}
param_lasso = {"alpha": np.logspace(-4, 1, 20)}
param_elasticnet = {"alpha": np.logspace(-4, 1, 20), "l1_ratio": np.linspace(0, 1, 20)}
param_randomforest = {"n_estimators": [50, 100], "max_depth": [None, 5, 10]}


#NOTE: AI GENERATED PARAMS
param_xgboost = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2]
}

param_gradientboost = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 7], 
    "learning_rate": [0.01, 0.1, 0.2]
}

param_adaboost = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1, 0.2]
}



models = {
    'ridge': Ridge(),
    'lasso': Lasso(),
    'elasticnet': ElasticNet(),
    'randomforest': RandomForestRegressor(random_state=42),
    'xgboost': XGBoost(),
    'gradientboost': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
}

param_grids = {
    'ridge': param_ridge,
    'lasso': param_lasso,
    'elasticnet': param_elasticnet,
    'randomforest': param_randomforest,
    'xgboost': param_xgboost,
    'gradientboost': param_gradientboost,
    'adaboost': param_adaboost,
}


X = train.drop(columns=['return_rate_pct'])
y = train['return_rate_pct']

In [ ]:




outer_scores = {name: [] for name in models.keys()}


optimal_threshold = threshold_finder.tree_.threshold[0]



for train_idx, val_idx in outer_cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_low_mask = X_train['price_eur'] <= optimal_threshold
    val_low_mask = X_val['price_eur'] <= optimal_threshold
    
    X_train_low, y_train_low = X_train[train_low_mask], y_train[train_low_mask]
    X_train_high, y_train_high = X_train[~train_low_mask], y_train[~train_low_mask]
    
    X_val_low, y_val_low = X_val[val_low_mask], y_val[val_low_mask]
    X_val_high, y_val_high = X_val[~val_low_mask], y_val[~val_low_mask]

    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    for name, model in models.items():
        # Tune on Low segment
        grid_low = GridSearchCV(model, param_grids[name], cv=inner_cv)
        grid_low.fit(X_train_low, y_train_low)
        
        # Tune on High segment
        grid_high = GridSearchCV(model, param_grids[name], cv=inner_cv)
        grid_high.fit(X_train_high, y_train_high)
        
        preds = np.zeros(len(X_val))
        
        # Predict using tuned models
        if len(X_val_low) > 0:
            preds[val_low_mask] = grid_low.best_estimator_.predict(X_val_low)
        if len(X_val_high) > 0:
            preds[~val_low_mask] = grid_high.best_estimator_.predict(X_val_high)
            
        fold_mse = mean_squared_error(y_val, preds)
        outer_scores[name].append(fold_mse)

for name, scores in outer_scores.items():
    print(f"{name} | Avg MSE: {np.mean(scores):.4f} | Std: {np.std(scores):.4f}")



ridge | Avg MSE: 57.1191 | Std: 1.8456
lasso | Avg MSE: 57.1578 | Std: 1.6153
elasticnet | Avg MSE: 57.3180 | Std: 1.9145
randomforest | Avg MSE: 28.3466 | Std: 1.8956
xgboost | Avg MSE: 26.1514 | Std: 1.7260
gradientboost | Avg MSE: 26.3494 | Std: 1.9257
adaboost | Avg MSE: 31.9165 | Std: 2.5186
svm | Avg MSE: 36.1346 | Std: 1.7042


Unfortunately the threshold separation seems to make the prediction even worse.

## THE CODE BELOW WAS A TEST WITHOUT NESTED

I have no idea if that could be usefull but these MSE were better than the previous Nested.

In [8]:
def print_mse(predictions_list, true_values):
    for name, predictions in predictions_list.items():
        mse = mean_squared_error(true_values, predictions)
        print(f"{name} MSE: {mse:.4f}")


In [11]:
test_predictions = {name: np.zeros(len(test)) for name in models.keys()}
is_low_test = test['price_eur'] <= optimal_threshold
is_high_test = test['price_eur'] > optimal_threshold
features = train.drop(columns=['return_rate_pct']).columns


for name, clf in models.items():
    if is_low_test.any():
        grid_low = GridSearchCV(clf, globals()[f"param_{name}"], cv=KFold(n_splits=3, shuffle=True, random_state=42))
        grid_low.fit(X_low, y_low) 
        
        test_predictions[name][is_low_test] = grid_low.predict(test.loc[is_low_test, features])
        
    if is_high_test.any():
        grid_high = GridSearchCV(clf, globals()[f"param_{name}"], cv=KFold(n_splits=3, shuffle=True, random_state=42))
        grid_high.fit(X_high, y_high) 
        
        test_predictions[name][is_high_test] = grid_high.predict(test.loc[is_high_test, features])


print_mse(test_predictions, test['return_rate_pct'])

KeyboardInterrupt: 

## Different Model Based on the Threshold

We did the Nested CV using the same two model for both the high and low ones. <br>
Instead, we'll try to find the best model for each part of the datas and try that instead.

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression


models['polynomial'] = Pipeline([
    ('poly', PolynomialFeatures(degree=4)), 
    ('scaler', StandardScaler()),
    ('linear', LinearRegression())
])

param_grids['polynomial'] = {
    'poly__degree': [i for i in range(2, 5)] # This will test linear (1), quadratic (2), and cubic (3)
}

In [27]:
print(param_grids.keys())

dict_keys(['ridge', 'lasso', 'elasticnet', 'randomforest', 'xgboost', 'gradientboost', 'adaboost', 'polynomial'])


## LOW SEGMENT

In [17]:
# Create a dictionary to store the results for each model in the Low segment
low_segment_results = {name: [] for name in models.keys()}

print("--- Evaluating Models for the LOW TIER ---")

# We use the same outer loop structure, but focus only on the LOW segment
for train_idx, val_idx in outer_cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Filter for LOW segment ONLY
    train_low_mask = X_train['price_eur'] <= optimal_threshold
    val_low_mask = X_val['price_eur'] <= optimal_threshold
    
    X_train_low, y_train_low = X_train[train_low_mask], y_train[train_low_mask]
    X_val_low, y_val_low = X_val[val_low_mask], y_val[val_low_mask]
    
    # Skip this fold if there isn't enough data
    if len(X_train_low) < 5 or len(X_val_low) == 0:
        continue

    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    for name, model in models.items():
        try:
            grid = GridSearchCV(model, param_grids[name], cv=inner_cv, scoring='neg_mean_squared_error')
            grid.fit(X_train_low, y_train_low)
            
            preds = grid.best_estimator_.predict(X_val_low)
            mse = mean_squared_error(y_val_low, preds)
            low_segment_results[name].append(mse)
        except Exception as e:
            print(f"CRASHED on {name}: {e}")
            continue # Skip the bad model instead of killing the whole kernel
    print(f"Completed fold with {len(X_val_low)} LOW samples.")
        
sort = sorted(low_segment_results.items(), key=lambda x: np.mean(x[1]))

# Report results
print("\nResults for Low Tier:")
for name, scores in sort:
    print(f"{name.upper()} | Avg MSE: {np.mean(scores):.4f} | Std: {np.std(scores):.4f}")

--- Evaluating Models for the LOW TIER ---
Completed fold with 156 LOW samples.
Completed fold with 154 LOW samples.
Completed fold with 168 LOW samples.
Completed fold with 161 LOW samples.
Completed fold with 157 LOW samples.

Results for Low Tier:
POLYNOMIAL | Avg MSE: 11.1262 | Std: 1.3777
LIGHTGBM | Avg MSE: 12.2459 | Std: 0.6208
GRADIENTBOOST | Avg MSE: 12.9048 | Std: 1.2395
XGBOOST | Avg MSE: 12.9585 | Std: 1.1199
SVM | Avg MSE: 16.3395 | Std: 0.9970
RANDOMFOREST | Avg MSE: 16.9490 | Std: 0.5331
ADABOOST | Avg MSE: 22.5563 | Std: 1.6061
LASSO | Avg MSE: 47.5869 | Std: 2.8103
RIDGE | Avg MSE: 47.5927 | Std: 2.8747
ELASTICNET | Avg MSE: 47.6357 | Std: 2.9108


The best Model for the low ones seems to be GradientBoost at 12.91 <br>
Testing XGBoost might also be a good idea considering how close their MSE are

## HIGH

In [18]:
# Create a dictionary to store the results for each model in the High segment
high_segment_results = {name: [] for name in models.keys()}

print("--- Evaluating Models for the HIGH TIER ---")

# We use the same outer loop structure
for train_idx, val_idx in outer_cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Filter for HIGH segment ONLY (using the tilde ~ to invert the mask)
    train_high_mask = X_train['price_eur'] > optimal_threshold
    val_high_mask = X_val['price_eur'] > optimal_threshold
    
    X_train_high, y_train_high = X_train[train_high_mask], y_train[train_high_mask]
    X_val_high, y_val_high = X_val[val_high_mask], y_val[val_high_mask]
    
    # Skip this fold if there isn't enough data
    if len(X_train_high) < 5 or len(X_val_high) == 0:
        continue

    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    for name, model in models.items():
        # Tune the model on the High data
        grid = GridSearchCV(model, param_grids[name], cv=inner_cv, scoring='neg_mean_squared_error')
        grid.fit(X_train_high, y_train_high)
        
        # Predict on the High Val data
        preds = grid.best_estimator_.predict(X_val_high)
        mse = mean_squared_error(y_val_high, preds)
        high_segment_results[name].append(mse)

sort = sorted(high_segment_results.items(), key=lambda x: np.mean(x[1]))

# Report results
print("\nResults for High Tier:")
for name, scores in sort:
    print(f"{name.upper()} | Avg MSE: {np.mean(scores):.4f} | Std: {np.std(scores):.4f}")

--- Evaluating Models for the HIGH TIER ---

Results for High Tier:
LIGHTGBM | Avg MSE: 45.9860 | Std: 2.3132
RANDOMFOREST | Avg MSE: 46.1363 | Std: 3.8940
XGBOOST | Avg MSE: 46.8142 | Std: 3.7720
ADABOOST | Avg MSE: 47.2622 | Std: 5.0510
GRADIENTBOOST | Avg MSE: 47.4151 | Std: 4.1664
POLYNOMIAL | Avg MSE: 57.8010 | Std: 7.4382
SVM | Avg MSE: 67.3266 | Std: 2.8844
LASSO | Avg MSE: 72.2456 | Std: 2.5822
ELASTICNET | Avg MSE: 72.4615 | Std: 2.7451
RIDGE | Avg MSE: 72.5465 | Std: 3.1780


The best Model for the low ones seems to be RandomForest.

# BEST ONE 

In [ ]:
params = {
    'gradientboost': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.2], 'max_depth': [3, 5, 7]},
    'randomforest': {'n_estimators': [50, 100], 'max_depth': [None, 5, 10]}
}

bimodal_mse_scores = []

for train_idx, val_idx in outer_cv.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_low_mask = X_train['price_eur'] <= optimal_threshold
    val_low_mask = X_val['price_eur'] <= optimal_threshold
    
    # GradientBoost for Low Price
    grid_low = GridSearchCV(GradientBoostingRegressor(random_state=42), params['gradientboost'], cv=3)
    grid_low.fit(X_train[train_low_mask], y_train[train_low_mask])
    
    # RandomForest for High Price
    grid_high = GridSearchCV(RandomForestRegressor(random_state=42), params['randomforest'], cv=3)
    grid_high.fit(X_train[~train_low_mask], y_train[~train_low_mask])
    
    preds = np.zeros(len(X_val))
    
    # Predict Low tier using GradientBoost
    if val_low_mask.any():
        preds[val_low_mask] = grid_low.best_estimator_.predict(X_val[val_low_mask])
        
    # Predict High tier using RandomForest
    if (~val_low_mask).any():
        preds[~val_low_mask] = grid_high.best_estimator_.predict(X_val[~val_low_mask])
            
    fold_mse = mean_squared_error(y_val, preds)
    bimodal_mse_scores.append(fold_mse)

print(f"Bimodal Strategy (GBM-Low / RF-High) MSE: {np.mean(bimodal_mse_scores):.4f} (± {np.std(bimodal_mse_scores):.4f})")

Bimodal Strategy (GBM-Low / RF-High) MSE: 25.6203 (± 2.2232)


This new Bimodal Strategy seems to have the lower MSE overall with GradientBoosting for low data and RandomForest for high data

## BELOW IS THE XGB TEST
The result is lower so idk if we should use it.

In [ ]:
from xgboost import XGBRegressor

params = {
    'xgboost': {"n_estimators": [50, 100], "max_depth": [3, 5, 7], "learning_rate": [0.01, 0.1, 0.2]},
    'randomforest': {'n_estimators': [50, 100], 'max_depth': [None, 5, 10]}
}


bimodal_mse_scores = []

for train_idx, val_idx in outer_cv.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_low_mask = X_train['price_eur'] <= optimal_threshold
    val_low_mask = X_val['price_eur'] <= optimal_threshold
    
    # XGBoost for Low Price
    grid_low = GridSearchCV(XGBRegressor(random_state=42), params['xgboost'], cv=3)
    grid_low.fit(X_train[train_low_mask], y_train[train_low_mask])
    
    # RandomForest for High Price
    grid_high = GridSearchCV(RandomForestRegressor(random_state=42), params['randomforest'], cv=3)
    grid_high.fit(X_train[~train_low_mask], y_train[~train_low_mask])
    
    preds = np.zeros(len(X_val))
    
    # Predict Low tier using XGBOOST
    if val_low_mask.any():
        preds[val_low_mask] = grid_low.best_estimator_.predict(X_val[val_low_mask])
        
    # Predict High tier using RandomForest
    if (~val_low_mask).any():
        preds[~val_low_mask] = grid_high.best_estimator_.predict(X_val[~val_low_mask])
            
    fold_mse = mean_squared_error(y_val, preds)
    bimodal_mse_scores.append(fold_mse)

print(f"Bimodal Strategy (XGBoost-Low / RandomForest-High) MSE: {np.mean(bimodal_mse_scores):.4f} (± {np.std(bimodal_mse_scores):.4f})")

Bimodal Strategy (XGBoost-Low / RandomForest-High) MSE: 25.8596 (± 2.2648)


## Poly

In [ ]:
#Randomforest 
model_randomforest = RandomForestRegressor(random_state=42)


model_poly = Pipeline([
    ('poly', PolynomialFeatures(degree=4)), 
    ('scaler', StandardScaler()),
    ('linear', LinearRegression())
])

poly_params = {
    'poly__degree': [i for i in range(0, 6)] # This will test linear (1), quadratic (2), and cubic (3)
}

bimodal_mse_scores = []

for train_idx, val_idx in outer_cv.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_low_mask = X_train['price_eur'] <= optimal_threshold
    val_low_mask = X_val['price_eur'] <= optimal_threshold
    
    # Polynomial for Low Price
    grid_low = GridSearchCV(model_poly, poly_params, cv=3)
    grid_low.fit(X_train[train_low_mask], y_train[train_low_mask])
    
    # Rf for High Price
    grid_high = GridSearchCV(model_randomforest, param_randomforest, cv=3)
    grid_high.fit(X_train[~train_low_mask], y_train[~train_low_mask])
    
    preds = np.zeros(len(X_val))
    
    # Predict Low tier using Polynomial
    if val_low_mask.any():
        preds[val_low_mask] = grid_low.best_estimator_.predict(X_val[val_low_mask])
        
    # Predict High tier using Rf
    if (~val_low_mask).any():
        preds[~val_low_mask] = grid_high.best_estimator_.predict(X_val[~val_low_mask])
            
    fold_mse = mean_squared_error(y_val, preds)
    bimodal_mse_scores.append(fold_mse)

print(f"Bimodal Strategy (Polynomial-Low / Rf-High) MSE: {np.mean(bimodal_mse_scores):.4f} (± {np.std(bimodal_mse_scores):.4f})")

Bimodal Strategy (Polynomial-Low / lightgbm-High) MSE: 24.7528 (± 2.2737)
